<a href="https://colab.research.google.com/github/Aymane-Aziz/toxic-comment-classifier/blob/main/04_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install wandb for experiment tracking
!pip install wandb -q

from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import f1_score, roc_auc_score
import wandb

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# All hyperparameters live in one place — easy to change and track
CONFIG = {
    'model_name'   : 'roberta-base',
    'max_len'      : 128,
    'batch_size'   : 32,
    'epochs'       : 3,
    'learning_rate': 2e-5,       # standard starting LR for transformers
    'warmup_steps' : 100,        # gradual LR warmup to stabilize early training
    'base_path'    : '/content/drive/MyDrive/toxic-comment-classifier/',
}

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Config ready, device:", DEVICE)

In [ ]:
class ToxicCommentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data      = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text   = str(self.data['cleaned_text'][idx])
        labels = self.data[LABEL_COLS].iloc[idx].values.astype(float)

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids'     : encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels'        : torch.tensor(labels, dtype=torch.float)
        }

In [ ]:
BASE_PATH = CONFIG['base_path'] + 'data/'

train_df = pd.read_csv(BASE_PATH + 'train_processed.csv')
val_df   = pd.read_csv(BASE_PATH + 'val_processed.csv')

# Fill NaNs just in case
train_df['cleaned_text'] = train_df['cleaned_text'].fillna('')
val_df['cleaned_text']   = val_df['cleaned_text'].fillna('')

tokenizer = RobertaTokenizer.from_pretrained(CONFIG['model_name'])

train_dataset = ToxicCommentDataset(train_df, tokenizer, CONFIG['max_len'])
val_dataset   = ToxicCommentDataset(val_df,   tokenizer, CONFIG['max_len'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=6,           # one output per label
    problem_type='multi_label_classification'
)

model = model.to(DEVICE)
print("Model loaded and moved to", DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Load saved class weights from Phase 3
pos_weights = torch.load(BASE_PATH + 'pos_weights.pt').to(DEVICE)

# BCEWithLogitsLoss = Binary Cross Entropy — perfect for multi-label problems
# pos_weight tells the loss to penalize missing rare labels more heavily
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# AdamW is the standard optimizer for transformers
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.01)

# Scheduler gradually reduces learning rate during training
total_steps = len(train_loader) * CONFIG['epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG['warmup_steps'],
    num_training_steps=total_steps
)

print("Loss, optimizer and scheduler ready")

In [ ]:
wandb.login()  # it will ask for your API key — get it from wandb.ai/settings

run = wandb.init(
    project='toxic-comment-classifier',
    config=CONFIG,
    name='roberta-base-run-1'
)

print("wandb initialized!")

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()               # clear previous gradients
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()                     # compute gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding gradients
        optimizer.step()                    # update weights
        scheduler.step()                    # update learning rate

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} — Loss: {loss.item():.4f}")

    return total_loss / len(loader)

In [ ]:
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():   # no gradient computation needed during evaluation
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.sigmoid(outputs.logits).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    # Convert probabilities to binary predictions at 0.5 threshold
    binary_preds = (all_preds > 0.5).astype(int)

    f1      = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
    roc_auc = roc_auc_score(all_labels, all_preds, average='macro')

    return total_loss / len(loader), f1, roc_auc

In [ ]:
best_roc_auc   = 0
SAVE_PATH      = CONFIG['base_path'] + 'checkpoints/'
os.makedirs(SAVE_PATH, exist_ok=True)

for epoch in range(CONFIG['epochs']):
    print(f"\n{'='*50}")
    print(f"EPOCH {epoch+1}/{CONFIG['epochs']}")
    print(f"{'='*50}")

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    val_loss, val_f1, val_roc_auc = eval_epoch(model, val_loader, criterion, DEVICE)

    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val F1:     {val_f1:.4f}")
    print(f"Val ROC-AUC:{val_roc_auc:.4f}")

    # Log to wandb
    wandb.log({
        'epoch'      : epoch + 1,
        'train_loss' : train_loss,
        'val_loss'   : val_loss,
        'val_f1'     : val_f1,
        'val_roc_auc': val_roc_auc
    })

    # Save best model
    if val_roc_auc > best_roc_auc:
        best_roc_auc = val_roc_auc
        model.save_pretrained(SAVE_PATH + 'best_model')
        tokenizer.save_pretrained(SAVE_PATH + 'best_model')
        print(f"✅ New best model saved! ROC-AUC: {best_roc_auc:.4f}")

wandb.finish()
print(f"\nTraining complete! Best ROC-AUC: {best_roc_auc:.4f}")